In [3]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import GPT2Model, GPT2Tokenizer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, accuracy_score

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2Model.from_pretrained("gpt2")
model.eval()

print("Ready!")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 4118.28it/s]

Ready!


In [4]:
import urllib.request

url = "https://www.gutenberg.org/files/2701/2701-0.txt"
with urllib.request.urlopen(url) as f:
    book_text = f.read().decode("utf-8")


sentences = [s.strip() for s in book_text.split(".") 
             if len(s.strip()) > 50][:200]

print(f"Total sentences: {len(sentences)}")
print(f"Example: {sentences[0][:80]}...")

Total sentences: 200
Example: *** START OF THE PROJECT GUTENBERG EBOOK 2701 ***




MOBY-DICK;

or, THE WHALE...


In [5]:
all_embeddings = []
all_positions  = []

for sentence in sentences:
    inputs = tokenizer(sentence, 
                      return_tensors="pt",
                      max_length=32,
                      truncation=True)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    embs = outputs.last_hidden_state[0]  # [n_tokens, 768]
    n    = embs.shape[0]
    
    for pos in range(n):
        all_embeddings.append(embs[pos].numpy())
        all_positions.append(pos)

X = np.array(all_embeddings)
y = np.array(all_positions)

print(f"Total tokens: {len(y)}")
print(f"X shape: {X.shape}")
print(f"Position range: {y.min()} to {y.max()}")

Total tokens: 5436
X shape: (5436, 768)
Position range: 0 to 31


In [6]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)

r2  = r2_score(y_test, y_pred)
mae = np.mean(np.abs(y_pred - y_test))

print(f"Linear Regression Results:")
print(f"  R² score: {r2:.4f}")
print(f"  MAE:      {mae:.4f} positions")
print(f"\nExample predictions vs actual:")
for i in range(5):
    print(f"  Actual: {y_test[i]:3d}  Predicted: {y_pred[i]:.1f}")

Linear Regression Results:
  R² score: 0.7395
  MAE:      3.3196 positions

Example predictions vs actual:
  Actual:  24  Predicted: 27.6
  Actual:  24  Predicted: 19.4
  Actual:  30  Predicted: 18.1
  Actual:  23  Predicted: 16.3
  Actual:  11  Predicted: 2.3


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Binary classification — early (0-15) vs late (16-31)
y_binary = (y >= 16).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary, test_size=0.2, random_state=42)

# Logistic Regression
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)

y_pred = log_reg.predict(X_test)
acc    = accuracy_score(y_test, y_pred)

print(f"Logistic Regression Results:")
print(f"  Accuracy: {acc*100:.2f}%")
print(f"\nRandom guess would be: 50.00%")
print(f"Our model:             {acc*100:.2f}%")

Logistic Regression Results:
  Accuracy: 87.50%

Random guess would be: 50.00%
Our model:             87.50%


c:\Users\devuser3\AppData\Roaming\uv\python\cpython-3.14.3-windows-x86_64-none\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [9]:
log_reg2 = LogisticRegression(max_iter=5000)
log_reg2.fit(X_train, y_train)
y_pred2 = log_reg2.predict(X_test)
acc2 = accuracy_score(y_test, y_pred2)
print(f"Improved Accuracy: {acc2*100:.2f}%")

Improved Accuracy: 86.95%


c:\Users\devuser3\AppData\Roaming\uv\python\cpython-3.14.3-windows-x86_64-none\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
from sklearn.preprocessing import StandardScaler

# Data scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

log_reg3 = LogisticRegression(max_iter=1000)
log_reg3.fit(X_train_scaled, y_train)
y_pred3 = log_reg3.predict(X_test_scaled)
acc3 = accuracy_score(y_test, y_pred3)
print(f"Scaled Accuracy: {acc3*100:.2f}%")

Scaled Accuracy: 87.13%



* **Machine Learning techniques** help in understanding the **internal behavior of LLMs**.

* GPT-2 embeddings contain **positional information**, not just semantic meaning.

* Using **Linear Regression** (Linear Regression), you can predict token position with about **74% accuracy**.

* Using **Logistic Regression** (Logistic Regression), you can classify tokens as **early vs late** in a sequence with around **87.5% accuracy**.
